# APT Detection with Graph Neural Networks

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import networkx as nx
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import hashlib
import json
from collections import defaultdict, deque
import time

class APTDetectionSystem:
    def __init__(self, temporal_window=3600):
        self.behavior_graph = nx.DiGraph()
        self.temporal_window = temporal_window
        self.anomaly_detector = IsolationForest(contamination=0.1, random_state=42)
        self.scaler = StandardScaler()
        self.attack_signatures = defaultdict(list)
        self.event_buffer = deque(maxlen=10000)

    def extract_behavioral_features(self, events):
        features = []
        for event in events:
            proc_hash = hashlib.md5(event['process'].encode()).hexdigest()[:8]
            net_entropy = self.calculate_entropy(event.get('network_data', ''))
            file_ops = len(event.get('file_operations', []))
            reg_ops = len(event.get('registry_operations', []))
            lateral_moves = event.get('lateral_movement_indicators', 0)
            privilege_escalation = event.get('privilege_escalation', 0)
            data_exfiltration = event.get('data_exfiltration_volume', 0)
            features.append([net_entropy, file_ops, reg_ops, lateral_moves, privilege_escalation, data_exfiltration])
        return np.array(features)

    def calculate_entropy(self, data):
        if not data: return 0
        byte_counts = np.bincount(np.frombuffer(data.encode(), dtype=np.uint8))
        probs = byte_counts / len(data)
        return -np.sum(probs * np.log2(probs + 1e-10))

    def build_attack_graph(self, events):
        for event in events:
            source = f"{event['source_ip']}:{event['process']}"
            target = f"{event['target_ip']}:{event.get('target_process', 'unknown')}"
            self.behavior_graph.add_edge(source, target, weight=event.get('severity', 1), timestamp=event['timestamp'])

    def detect_kill_chain_patterns(self):
        kill_chain_stages = ['reconnaissance', 'weaponization', 'delivery', 'exploitation', 'installation', 'command_control', 'actions']
        detected_chains = []
        for component in nx.weakly_connected_components(self.behavior_graph):
            subgraph = self.behavior_graph.subgraph(component)
            if len(subgraph.nodes()) > 3:
                centrality = nx.betweenness_centrality(subgraph)
                max_centrality = max(centrality.values()) if centrality else 0
                if max_centrality > 0.5:
                    detected_chains.append({'nodes': list(component), 'centrality': max_centrality})
        return detected_chains

    def analyze_temporal_patterns(self, events):
        time_buckets = defaultdict(list)
        for event in events:
            bucket = int(event['timestamp'] // self.temporal_window)
            time_buckets[bucket].append(event)

        anomalous_periods = []
        for bucket, bucket_events in time_buckets.items():
            if len(bucket_events) > 0:
                features = self.extract_behavioral_features(bucket_events)
                if features.size > 0:
                    features_scaled = self.scaler.fit_transform(features)
                    anomalies = self.anomaly_detector.fit_predict(features_scaled)
                    anomaly_ratio = np.sum(anomalies == -1) / len(anomalies)
                    if anomaly_ratio > 0.3:
                        anomalous_periods.append({'bucket': bucket, 'anomaly_ratio': anomaly_ratio})
        return anomalous_periods

    def detect_apt_campaign(self, events):
        self.build_attack_graph(events)
        kill_chains = self.detect_kill_chain_patterns()
        temporal_anomalies = self.analyze_temporal_patterns(events)

        campaign_score = len(kill_chains) * 0.4 + len(temporal_anomalies) * 0.3
        campaign_score += min(len(self.behavior_graph.nodes()) / 100, 0.3)

        return {'score': campaign_score, 'kill_chains': kill_chains, 'temporal_anomalies': temporal_anomalies, 'threat_level': 'HIGH' if campaign_score > 0.7 else 'MEDIUM' if campaign_score > 0.4 else 'LOW'}